# Pydantic格式的使用

## 基本使用

举例1：

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model = "deepseek-flash",
    model_provider = "deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL,
        extra_body = {
        "thinking":{"type":"disabled"}
    }
)

In [ ]:
from pydantic import BaseModel, Field
from rich import print as rprint

class Person(BaseModel):
    """人物的信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

res = structured_model.invoke("张三是一名30岁的软件工程师")
rprint(res)
rprint(type(res))

In [ ]:
rprint(f"姓名:{res.name}")
rprint(f"年龄:{res.age}")
rprint(f"职业:{res.occupation}")

举例2：

In [ ]:
class MovieModel(BaseModel):
    """电影的详细信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="发行年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")

structured_model = model.with_structured_output(MovieModel)


res = structured_model.invoke("给出电影盗梦空间的信息")
rprint(res)

举例3：

In [ ]:
from pydantic import BaseModel, Field

class SentimentAnalysis(BaseModel):
    """情感分析结果"""
    sentiment: str = Field(description="情感倾向：positive/negative/neutral")
    confidence: float = Field(description="置信度：0-1之间")
    keywords: list[str] = Field(description="关键词列表")

structured_model = model.with_structured_output(SentimentAnalysis)

text = "这个课程很有用，学到了很多东西，强烈推荐"
res = structured_model.invoke(
    f"分析一下文本的情感：\n{text}"
)

rprint(f"类型:{type(res)}")
rprint(f"情感:{res.sentiment}")
rprint(f"置信度:{res.confidence}")
rprint(f"关键词:{res.keywords}")

## 2.高级特性

### 2.1 情况1：可选字段

举例：

In [ ]:
from pydantic import BaseModel, Field
from rich import print as rprint

class Person(BaseModel):
    """人物的信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

res = structured_model.invoke("张三是一名软件工程师")
rprint(res)
rprint(type(res))

作为对比：

In [ ]:
from pydantic import BaseModel, Field
from rich import print as rprint
from typing import Optional

class Person(BaseModel):
    """人物的信息"""
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

res = structured_model.invoke("张三是一名软件工程师")
rprint(res)
rprint(type(res))

### 2.2 情况2：默认值

不同的模型供应商对于此字段的支持是不同的。

举例1：使用deepseek模型

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from rich import print as rprint

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model = "deepseek-flash",
    model_provider = "deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL,
        extra_body = {
        "thinking":{"type":"disabled"}
    }
)

class Person(BaseModel):
    """人物的信息"""
    name: str = Field(description="姓名")
    age: int = Field(default = 10, description="年龄")  # 默认值default = 10
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

res = structured_model.invoke("张三是一名软件工程师")
rprint(res)
rprint(type(res))

In [ ]:
class Person(BaseModel):
    """人物的信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄(默认值为20)")  # 相当于提示词
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

res = structured_model.invoke("张三是一名软件工程师")
rprint(res)
rprint(type(res))

### 2.3 情况3：枚举类型

方式一：

In [ ]:
from typing import Optional
from pydantic import Field
from enum import Enum

# 定义枚举类型
class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"

class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")

# 测试
structured_model = model.with_structured_output(CustomerInfo)

conversation = """
客服：您好，请问有什么可以帮助您？
客户：我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服：好的，我帮您查一下
"""

result = structured_model.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")

print(result)

print("\n提取结果：")
print(f"  客户：{result.name}")
print(f"  电话：{result.phone}")
print(f"  邮箱：{result.email or '未提供'}")
print(f"  问题：{result.issue}")
print(f"  紧急程度：{result.urgency.value}")

方式二：

In [29]:
from typing import Optional, Literal
from pydantic import Field

class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Literal["低", "中", "高"] = Field(description="紧急程度")

# 测试
structured_model = model.with_structured_output(CustomerInfo)

conversation = """
客服：您好，请问有什么可以帮助您？
客户：我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服：好的，我帮您查一下
"""

result = structured_model.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")

print(result)

print("\n提取结果：")
print(f"  客户：{result.name}")
print(f"  电话：{result.phone}")
print(f"  邮箱：{result.email or '未提供'}")
print(f"  问题：{result.issue}")

name='王小明' phone='138-1234-5678' email=None issue='订单一直未发货' urgency='高'

提取结果：
  客户：王小明
  电话：138-1234-5678
  邮箱：未提供
  问题：订单一直未发货
